In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Set display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Customer Purchase Data

In [ ]:
# Create sample customer purchase data
np.random.seed(42)

# Generate customer IDs
customers = [f"Customer_{i:03d}" for i in range(1, 101)]

# Generate product IDs
products = [
    "Laptop", "Smartphone", "Headphones", "Tablet", "Smartwatch",
    "Camera", "Gaming_Console", "Bluetooth_Speaker", "Keyboard", "Mouse",
    "Monitor", "Printer", "Router", "Hard_Drive", "USB_Cable",
    "Phone_Case", "Screen_Protector", "Charger", "Power_Bank", "Webcam"
]

# Create purchase matrix (customers x products)
# 1 = purchased, 0 = not purchased
purchase_data = []

for customer in customers:
    # Each customer purchases 3-8 random products
    num_purchases = np.random.randint(3, 9)
    purchased_products = np.random.choice(products, num_purchases, replace=False)
    
    for product in purchased_products:
        purchase_data.append({
            'customer_id': customer,
            'product_id': product,
            'purchased': 1
        })

# Convert to DataFrame
df_purchases = pd.DataFrame(purchase_data)

In [ ]:
# Create customer-product purchase matrix
purchase_matrix = df_purchases.pivot_table(
    index='customer_id',
    columns='product_id',
    values='purchased',
    fill_value=0
)

In [ ]:
# Check matrix sparsity
total_cells = purchase_matrix.shape[0] * purchase_matrix.shape[1]
filled_cells = (purchase_matrix > 0).sum().sum()
sparsity = (1 - filled_cells / total_cells) * 100

print(f"\nMatrix sparsity: {sparsity:.2f}%")

Rating-Based Dataset

In [ ]:
# Create sample rating data (1-5 scale)
rating_data = []

for customer in customers:
    # Each customer rates 5-12 random products
    num_ratings = np.random.randint(5, 13)
    rated_products = np.random.choice(products, num_ratings, replace=False)
    
    for product in rated_products:
        # Generate ratings with some bias (more 3-5 ratings)
        rating = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.1, 0.25, 0.35, 0.25])
        rating_data.append({
            'customer_id': customer,
            'product_id': product,
            'rating': rating
        })

# Convert to DataFrame
df_ratings = pd.DataFrame(rating_data)

# Create customer-product rating matrix
rating_matrix = df_ratings.pivot_table(
    index='customer_id',
    columns='product_id',
    values='rating',
    fill_value=0
)

In [ ]:
Implement Cosine Similarity Recommendations

In [ ]:
# Calculate cosine similarity between customers
def calculate_user_similarity(matrix):
    """Calculate cosine similarity between users (customers)"""
    # Convert to sparse matrix for efficiency
    sparse_matrix = csr_matrix(matrix.values)
    
    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(sparse_matrix)
    
    # Convert back to DataFrame for easier handling
    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=matrix.index,
        columns=matrix.index
    )
    
    return similarity_df

# Calculate similarity for purchase data
user_similarity_purchase = calculate_user_similarity(purchase_matrix)

print("User Similarity Matrix (Purchase-based):")
print(f"Shape: {user_similarity_purchase.shape}")
print("\nSimilarity between first 5 customers:")
print(user_similarity_purchase.iloc[:5, :5])

# Find most similar customers for a specific customer
def find_similar_customers(customer_id, similarity_matrix, top_n=5):
    """Find top N most similar customers"""
    if customer_id not in similarity_matrix.index:
        return f"Customer {customer_id} not found"
    
    # Get similarity scores for the customer (excluding self)
    similarities = similarity_matrix[customer_id].drop(customer_id)
    
    # Sort and get top N
    top_similar = similarities.nlargest(top_n)
    
    return top_similar

# Example: Find similar customers
example_customer = "Customer_001"
similar_customers = find_similar_customers(example_customer, user_similarity_purchase)

print(f"\nTop 5 customers similar to {example_customer}:")
for customer, similarity in similar_customers.items():
    print(f"{customer}: {similarity:.3f}")

In [ ]:
# User based recommendation
def get_user_based_recommendations(customer_id, purchase_matrix, similarity_matrix, top_n=5):
    """Generate recommendations based on similar customers"""
    
    if customer_id not in purchase_matrix.index:
        return f"Customer {customer_id} not found"
    
    # Get customer's current purchases
    customer_purchases = purchase_matrix.loc[customer_id]
    
    # Find similar customers (excluding self)
    similar_customers = similarity_matrix[customer_id].drop(customer_id)
    
    # Get top 10 most similar customers
    top_similar = similar_customers.nlargest(10)
    
    # Calculate weighted scores for products
    product_scores = {}
    
    for product in purchase_matrix.columns:
        # Skip if customer already purchased this product
        if customer_purchases[product] > 0:
            continue
            
        # Calculate weighted score based on similar customers
        weighted_score = 0
        total_weight = 0
        
        for similar_customer, similarity_score in top_similar.items():
            if purchase_matrix.loc[similar_customer, product] > 0:
                weighted_score += similarity_score
                total_weight += similarity_score
        
        if total_weight > 0:
            product_scores[product] = weighted_score / total_weight
    
    # Sort products by score and return top N
    recommended_products = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)
    
    return recommended_products[:top_n]

# Generate recommendations for example customer
recommendations = get_user_based_recommendations(
    example_customer, 
    purchase_matrix, 
    user_similarity_purchase, 
    top_n=5
)

print(f"\nTop 5 product recommendations for {example_customer}:")
for i, (product, score) in enumerate(recommendations, 1):
    print(f"{i}. {product}: {score:.3f}")

# Show what the customer has already purchased
current_purchases = purchase_matrix.loc[example_customer]
purchased_products = current_purchases[current_purchases > 0].index.tolist()
print(f"\n{example_customer} has already purchased:")
print(purchased_products)

In [ ]:
#  Item-Based Cosine Similarity
def calculate_item_similarity(matrix):
    """Calculate cosine similarity between items (products)"""
    # Transpose matrix so products are rows
    item_matrix = matrix.T
    
    # Convert to sparse matrix
    sparse_matrix = csr_matrix(item_matrix.values)
    
    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(sparse_matrix)
    
    # Convert back to DataFrame
    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=item_matrix.index,
        columns=item_matrix.index
    )
    
    return similarity_df

# Calculate item similarity
item_similarity = calculate_item_similarity(purchase_matrix)

print("Item Similarity Matrix:")
print(f"Shape: {item_similarity.shape}")
print("\nSimilarity between first 5 products:")
print(item_similarity.iloc[:5, :5])

def get_item_based_recommendations(customer_id, purchase_matrix, item_similarity, top_n=5):
    """Generate recommendations based on similar items"""
    
    if customer_id not in purchase_matrix.index:
        return f"Customer {customer_id} not found"
    
    # Get customer's purchases
    customer_purchases = purchase_matrix.loc[customer_id]
    purchased_products = customer_purchases[customer_purchases > 0].index.tolist()
    
    # Calculate scores for all products
    product_scores = {}
    
    for product in purchase_matrix.columns:
        # Skip if already purchased
        if product in purchased_products:
            continue
        
        # Calculate score based on similarity to purchased products
        score = 0
        for purchased_product in purchased_products:
            score += item_similarity.loc[product, purchased_product]
        
        if len(purchased_products) > 0:
            product_scores[product] = score / len(purchased_products)
    
    # Sort and return top N
    recommended_products = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)
    
    return recommended_products[:top_n]

# Generate item-based recommendations
item_recommendations = get_item_based_recommendations(
    example_customer,
    purchase_matrix,
    item_similarity,
    top_n=5
)

print(f"\nTop 5 item-based recommendations for {example_customer}:")
for i, (product, score) in enumerate(item_recommendations, 1):
    print(f"{i}. {product}: {score:.3f}")

Implement NearestNeighbors Recommendations

In [ ]:
from sklearn.neighbors import NearestNeighbors

def setup_user_knn_model(matrix, n_neighbors=6, metric='cosine'):
    """Set up KNN model for user-based recommendations"""
    
    # Create and fit the model
    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric=metric,
        algorithm='brute'  # Use brute force for cosine similarity
    )
    
    # Fit the model with user data
    model.fit(matrix.values)
    
    return model

def setup_item_knn_model(matrix, n_neighbors=6, metric='cosine'):
    """Set up KNN model for item-based recommendations"""
    
    # Transpose matrix for item-based similarity
    item_matrix = matrix.T
    
    # Create and fit the model
    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric=metric,
        algorithm='brute'
    )
    
    # Fit the model with item data
    model.fit(item_matrix.values)
    
    return model

# Set up models
user_knn_model = setup_user_knn_model(purchase_matrix)
item_knn_model = setup_item_knn_model(purchase_matrix)

In [ ]:
def get_knn_user_recommendations(customer_id, purchase_matrix, knn_model, top_n=5):
    """Generate recommendations using KNN user-based approach"""
    
    if customer_id not in purchase_matrix.index:
        return f"Customer {customer_id} not found"
    
    # Get customer index
    customer_idx = purchase_matrix.index.get_loc(customer_id)
    
    # Get customer's purchase vector
    customer_vector = purchase_matrix.iloc[customer_idx].values.reshape(1, -1)
    
    # Find similar customers
    distances, indices = knn_model.kneighbors(customer_vector)
    
    # Get similar customers (excluding self - first index)
    similar_indices = indices[0][1:]  # Skip first (self)
    similar_distances = distances[0][1:]
    
    # Convert distances to similarities (1 - distance for cosine)
    similarities = 1 - similar_distances
    
    # Get customer's current purchases
    customer_purchases = purchase_matrix.iloc[customer_idx]
    
    # Calculate product scores
    product_scores = {}
    
    for product_idx, product in enumerate(purchase_matrix.columns):
        # Skip if already purchased
        if customer_purchases.iloc[product_idx] > 0:
            continue
        
        # Calculate weighted score
        weighted_score = 0
        total_weight = 0
        
        for sim_idx, similarity in zip(similar_indices, similarities):
            if purchase_matrix.iloc[sim_idx, product_idx] > 0:
                weighted_score += similarity
                total_weight += similarity
        
        if total_weight > 0:
            product_scores[product] = weighted_score / total_weight
    
    # Sort and return top N
    recommended_products = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)
    
    return recommended_products[:top_n]

# Generate KNN user-based recommendations
knn_user_recommendations = get_knn_user_recommendations(
    example_customer,
    purchase_matrix,
    user_knn_model,
    top_n=5
)

print(f"\nTop 5 KNN user-based recommendations for {example_customer}:")
for i, (product, score) in enumerate(knn_user_recommendations, 1):
    print(f"{i}. {product}: {score:.3f}")

In [ ]:
def get_knn_item_recommendations(customer_id, purchase_matrix, knn_model, top_n=5):
    """Generate recommendations using KNN item-based approach"""
    
    if customer_id not in purchase_matrix.index:
        return f"Customer {customer_id} not found"
    
    # Get customer's purchases
    customer_purchases = purchase_matrix.loc[customer_id]
    purchased_products = customer_purchases[customer_purchases > 0].index.tolist()
    
    if not purchased_products:
        return "Customer has no purchase history"
    
    # Calculate scores for all products
    product_scores = {}
    
    for product in purchase_matrix.columns:
        # Skip if already purchased
        if product in purchased_products:
            continue
        
        # Get product index in transposed matrix
        product_idx = purchase_matrix.columns.get_loc(product)
        product_vector = purchase_matrix.T.iloc[product_idx].values.reshape(1, -1)
        
        # Find similar products
        distances, indices = knn_model.kneighbors(product_vector)
        
        # Calculate similarity score based on purchased products
        score = 0
        count = 0
        
        for similar_idx, distance in zip(indices[0], distances[0]):
            similar_product = purchase_matrix.columns[similar_idx]
            if similar_product in purchased_products:
                similarity = 1 - distance  # Convert distance to similarity
                score += similarity
                count += 1
        
        if count > 0:
            product_scores[product] = score / count
    
    # Sort and return top N
    recommended_products = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)
    
    return recommended_products[:top_n]

# Generate KNN item-based recommendations
knn_item_recommendations = get_knn_item_recommendations(
    example_customer,
    purchase_matrix,
    item_knn_model,
    top_n=5
)

print(f"\nTop 5 KNN item-based recommendations for {example_customer}:")
for i, (product, score) in enumerate(knn_item_recommendations, 1):
    print(f"{i}. {product}: {score:.3f}")

Generate Top-N Recommendations for All Users

In [ ]:
def generate_recommendations_for_all_users(purchase_matrix, method='user_cosine', top_n=3):
    """Generate recommendations for all users using specified method"""
    
    all_recommendations = {}
    
    if method == 'user_cosine':
        # Calculate user similarity once
        user_similarity = calculate_user_similarity(purchase_matrix)
        
        for customer in purchase_matrix.index:
            recommendations = get_user_based_recommendations(
                customer, purchase_matrix, user_similarity, top_n
            )
            all_recommendations[customer] = recommendations
            
    elif method == 'item_cosine':
        # Calculate item similarity once
        item_similarity = calculate_item_similarity(purchase_matrix)
        
        for customer in purchase_matrix.index:
            recommendations = get_item_based_recommendations(
                customer, purchase_matrix, item_similarity, top_n
            )
            all_recommendations[customer] = recommendations
            
    elif method == 'knn_user':
        # Set up KNN model once
        knn_model = setup_user_knn_model(purchase_matrix)
        
        for customer in purchase_matrix.index:
            recommendations = get_knn_user_recommendations(
                customer, purchase_matrix, knn_model, top_n
            )
            all_recommendations[customer] = recommendations
    
    return all_recommendations

# Generate recommendations for first 10 customers using different methods
sample_customers = purchase_matrix.index[:10]

print("Generating recommendations for sample customers...\n")

# User-based cosine similarity
print("=== USER-BASED COSINE SIMILARITY ===")
user_cosine_recs = {}
user_similarity = calculate_user_similarity(purchase_matrix)

for customer in sample_customers:
    recs = get_user_based_recommendations(customer, purchase_matrix, user_similarity, 3)
    user_cosine_recs[customer] = recs
    print(f"{customer}: {[product for product, score in recs]}")

print("\n=== ITEM-BASED COSINE SIMILARITY ===")
item_cosine_recs = {}
item_similarity = calculate_item_similarity(purchase_matrix)

for customer in sample_customers:
    recs = get_item_based_recommendations(customer, purchase_matrix, item_similarity, 3)
    item_cosine_recs[customer] = recs
    print(f"{customer}: {[product for product, score in recs]}")

print("\n=== KNN USER-BASED ===")
knn_user_recs = {}
knn_model = setup_user_knn_model(purchase_matrix)

for customer in sample_customers:
    recs = get_knn_user_recommendations(customer, purchase_matrix, knn_model, 3)
    knn_user_recs[customer] = recs
    print(f"{customer}: {[product for product, score in recs]}")

In [ ]:
def create_recommendation_summary(recommendations_dict, method_name):
    """Create a summary of recommendations"""
    
    # Count product frequency in recommendations
    product_counts = {}
    total_customers = len(recommendations_dict)
    
    for customer, recs in recommendations_dict.items():
        for product, score in recs:
            if product not in product_counts:
                product_counts[product] = 0
            product_counts[product] += 1
    
    # Sort by frequency
    popular_recommendations = sorted(
        product_counts.items(), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    print(f"\n=== {method_name} SUMMARY ===")
    print(f"Total customers: {total_customers}")
    print(f"Most frequently recommended products:")
    
    for i, (product, count) in enumerate(popular_recommendations[:10], 1):
        percentage = (count / total_customers) * 100
        print(f"{i:2d}. {product}: {count} customers ({percentage:.1f}%)")
    
    return popular_recommendations

# Create summaries
user_cosine_summary = create_recommendation_summary(user_cosine_recs, "USER-BASED COSINE")
item_cosine_summary = create_recommendation_summary(item_cosine_recs, "ITEM-BASED COSINE")
knn_user_summary = create_recommendation_summary(knn_user_recs, "KNN USER-BASED")

Evaluate Recommendation System Performance

In [ ]:
def calculate_recommendation_coverage(recommendations_dict, total_products):
    """Calculate catalog coverage - percentage of products recommended"""
    
    recommended_products = set()
    
    for customer, recs in recommendations_dict.items():
        for product, score in recs:
            recommended_products.add(product)
    
    coverage = len(recommended_products) / total_products * 100
    
    return coverage, recommended_products

def calculate_recommendation_diversity(recommendations_dict):
    """Calculate average diversity of recommendations per user"""
    
    diversities = []
    
    for customer, recs in recommendations_dict.items():
        if len(recs) > 1:
            # Simple diversity: number of unique categories/types
            # For this example, we'll use product name diversity
            products = [product for product, score in recs]
            diversity = len(set(products)) / len(products)
            diversities.append(diversity)
    
    avg_diversity = np.mean(diversities) if diversities else 0
    
    return avg_diversity

# Evaluate different methods
total_products = len(purchase_matrix.columns)

print("=== RECOMMENDATION SYSTEM EVALUATION ===\n")

methods = [
    ("User-Based Cosine", user_cosine_recs),
    ("Item-Based Cosine", item_cosine_recs),
    ("KNN User-Based", knn_user_recs)
]

for method_name, recs in methods:
    coverage, recommended_products = calculate_recommendation_coverage(recs, total_products)
    diversity = calculate_recommendation_diversity(recs)
    
    print(f"{method_name}:")
    print(f"  Catalog Coverage: {coverage:.1f}% ({len(recommended_products)}/{total_products} products)")
    print(f"  Average Diversity: {diversity:.3f}")
    print(f"  Products Recommended: {sorted(list(recommended_products))}")
    print()

In [ ]:
def analyze_recommendation_quality(customer_id, purchase_matrix, recommendations_dict):
    """Analyze the quality of recommendations for a specific customer"""
    
    if customer_id not in purchase_matrix.index:
        return "Customer not found"
    
    # Get customer's purchase history
    customer_purchases = purchase_matrix.loc[customer_id]
    purchased_products = customer_purchases[customer_purchases > 0].index.tolist()
    
    print(f"=== RECOMMENDATION ANALYSIS FOR {customer_id} ===")
    print(f"Customer's purchase history: {purchased_products}")
    print()
    
    for method_name, recs_dict in recommendations_dict.items():
        if customer_id in recs_dict:
            recommendations = recs_dict[customer_id]
            print(f"{method_name} Recommendations:")
            
            for i, (product, score) in enumerate(recommendations, 1):
                print(f"  {i}. {product} (score: {score:.3f})")
            
            print()

# Analyze recommendations for example customer
recommendation_methods = {
    "User-Based Cosine": user_cosine_recs,
    "Item-Based Cosine": item_cosine_recs,
    "KNN User-Based": knn_user_recs
}

analyze_recommendation_quality(example_customer, purchase_matrix, recommendation_methods)

Advanced Features and Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_recommendation_patterns(recommendations_dict, method_name):
    """Visualize recommendation patterns"""
    
    # Count product recommendations
    product_counts = {}
    for customer, recs in recommendations_dict.items():
        for product, score in recs:
            if product not in product_counts:
                product_counts[product] = 0
            product_counts[product] += 1
    
    # Create visualization
    plt.figure(figsize=(12, 6))
    
    products = list(product_counts.keys())
    counts = list(product_counts.values())
    
    # Sort by count
    sorted_data = sorted(zip(products, counts), key=lambda x: x[1], reverse=True)
    products_sorted = [x[0] for x in sorted_data]
    counts_sorted = [x[1] for x in sorted_data]
    
    plt.bar(range(len(products_sorted)), counts_sorted)
    plt.title(f'Product Recommendation Frequency - {method_name}')
    plt.xlabel('Products')
    plt.ylabel('Number of Recommendations')
    plt.xticks(range(len(products_sorted)), products_sorted, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return product_counts

# Visualize patterns for user-based cosine method
print("Generating visualization for User-Based Cosine recommendations...")
user_cosine_patterns = visualize_recommendation_patterns(user_cosine_recs, "User-Based Cosine")

In [ ]:
def create_recommendation_dashboard(customer_id, purchase_matrix, all_methods):
    """Create a comprehensive dashboard for a customer's recommendations"""
    
    print("=" * 60)
    print(f"RECOMMENDATION DASHBOARD FOR {customer_id}")
    print("=" * 60)
    
    # Customer profile
    customer_purchases = purchase_matrix.loc[customer_id]
    purchased_products = customer_purchases[customer_purchases > 0].index.tolist()
    
    print(f"\n📊 CUSTOMER PROFILE:")
    print(f"   Total purchases: {len(purchased_products)}")
    print(f"   Products purchased: {purchased_products}")
    
    # Recommendations from all methods
    print(f"\n🎯 RECOMMENDATIONS:")
    
    all_recommendations = set()
    
    for method_name, recs_dict in all_methods.items():
        if customer_id in recs_dict:
            recommendations = recs_dict[customer_id]
            print(f"\n   {method_name}:")
            
            for i, (product, score) in enumerate(recommendations, 1):
                print(f"      {i}. {product} (confidence: {score:.3f})")
                all_recommendations.add(product)
    
    # Consensus recommendations
    method_counts = {}
    for method_name, recs_dict in all_methods.items():
        if customer_id in recs_dict:
            for product, score in recs_dict[customer_id]:
                if product not in method_counts:
                    method_counts[product] = 0
                method_counts[product] += 1
    
    consensus_recs = [(product, count) for product, count in method_counts.items() if count > 1]
    consensus_recs.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n🤝 CONSENSUS RECOMMENDATIONS (agreed by multiple methods):")
    if consensus_recs:
        for product, count in consensus_recs:
            print(f"   • {product} (recommended by {count} methods)")
    else:
        print("   No consensus recommendations found")
    
    print("\n" + "=" * 60)

# Create dashboard for example customer
create_recommendation_dashboard(example_customer, purchase_matrix, recommendation_methods)

# Create dashboard for another customer
another_customer = "Customer_005"
create_recommendation_dashboard(another_customer, purchase_matrix, recommendation_methods)